# Simple MA Crossover Backtest (MetaTrader5)

Este notebook demonstra como usar os mapeamentos do MetaTrader5 em `algo_trading/sources/MetaTrader5_source/` com os handlers genéricos de conta/posições para rodar um backtest simples de crossover de médias.

Etapas:
1. Login no MetaTrader5 (live)
2. Criação de uma conta de backtest
3. Obtenção de candles (Rates)
4. Execução do backtest de crossover (MA rápido vs lento)


In [3]:
# Imports principais
from datetime import datetime, timezone, timedelta
import pandas as pd

# Mapeamentos MetaTrader5
from algo_trading.sources.MetaTrader5_source.account.account import Account
from algo_trading.sources.MetaTrader5_source.rates.rates import Rates
from algo_trading.sources.MetaTrader5_source.models.enums import ENUM_TIMEFRAME



## 1) Login no MetaTrader5 (live)
Preencha as credenciais do servidor abaixo. O `login_backtest` exige uma sessão live inicializada (por design).


In [4]:
# Substitua com suas credenciais do MT5
LOGIN = 5041151393             # int
SERVER = "MetaQuotes-Demo" # str (ex.: "MetaQuotes-Demo")
PASSWORD = "V*Fo2qVq"  # str
MT5_PATH = ""                 # opcional: caminho do terminal

account = Account()
# Isto inicializa mt5.initialize(...) internamente
_live = account.login_live(login=LOGIN, server=SERVER, password=PASSWORD, path=MT5_PATH)
_live


2025-10-11 16:01:02,860 - INFO - Successfully logged in to live account #5041151393
2025-10-11 16:01:02,875 - INFO - Live account data successfully updated.


MqlAccountInfo(login=5041151393, trade_mode=0, leverage=100, limit_orders=200, margin_so_mode=0, trade_allowed=True, trade_expert=True, margin_mode=2, currency_digits=2, fifo_close=False, balance=100000.0, credit=0.0, profit=0.0, equity=100000.0, margin=0.0, margin_free=100000.0, margin_level=0.0, margin_so_call=50.0, margin_so_so=30.0, margin_initial=0.0, margin_maintenance=0.0, assets=0.0, liabilities=0.0, commission_blocked=0.0, name='Pedro Bento', server='MetaQuotes-Demo', currency='USD', company='MetaQuotes Ltd.', orders=[], positions=[], history_deals=[MqlTradeDeal(ticket=53416955508, order=0, time=datetime.datetime(2025, 10, 11, 21, 33, 57, tzinfo=datetime.timezone.utc), time_msc=datetime.datetime(2025, 10, 11, 21, 33, 57, 105000, tzinfo=datetime.timezone.utc), type=2, entry=0, magic=0, position_id=0, reason=0, volume=0.0, price=0.0, commission=0.0, swap=0.0, profit=100000.0, fee=0.0, symbol='', comment='', external_id='')], is_backtest_account=False, rates_data=<class 'algo_tra

## 2) Criar conta de Backtest


In [5]:
backtest = account.login_backtest(balance=10_000, leverage=100)
backtest


2025-10-11 16:01:04,574 - INFO - Backtest account successfully created.


MqlAccountInfo(login=9999999, trade_mode=0, leverage=100, limit_orders=200, margin_so_mode=0, trade_allowed=True, trade_expert=True, margin_mode=2, currency_digits=2, fifo_close=False, balance=10000.0, credit=0.0, profit=0.0, equity=10000.0, margin=0.0, margin_free=10000.0, margin_level=0.0, margin_so_call=50.0, margin_so_so=30.0, margin_initial=0.0, margin_maintenance=0.0, assets=0.0, liabilities=0.0, commission_blocked=0.0, name='Backtest Account', server='Backtest Server', currency='USD', company='Backtest Company', orders=[], positions=[], history_deals=[MqlTradeDeal(ticket=1760209264570439, order=0, time=datetime.datetime(2025, 10, 11, 19, 1, 4, tzinfo=datetime.timezone.utc), time_msc=datetime.datetime(2025, 10, 11, 19, 1, 4, 570439, tzinfo=datetime.timezone.utc), type=2, entry=0, magic=0, position_id=0, reason=3, volume=0.0, price=0.0, commission=0.0, swap=0.0, profit=10000.0, fee=0.0, symbol='', comment='', external_id=None)], is_backtest_account=True, rates_data=<class 'algo_tr

In [ ]:
backtest.operation.

,tick_size,contract_size,trade_calc_mode,swap_mode,swap_long,swap_short,volume_min,volume_max,volume_step,volume_limit,swap_rollover3days,last_candle
symbol,,,,,,,,,,,,
EURUSD,0.00001,100000.0,0,1,-0.7,-1.0,0.01,500.0,0.01,0.0,3,None


## 3) Obter candles via `Rates.get_candles_range()`


In [5]:
symbol = "EURUSD"
timeframe = ENUM_TIMEFRAME.TIMEFRAME_M5
date_to = datetime.now(timezone.utc)
date_from = date_to - timedelta(days=30)

ohlc = Rates.get_candles_range(
    symbol=symbol,
    date_from=date_from,
    date_to=date_to,
    timeframe=timeframe,
    use_close_candle_time=False,
)
display(ohlc.head())
print("Rows:", len(ohlc))


,open,high,low,close,tick_volume
time,,,,,
2025-09-11 18:35:00+00:00,1.17371,1.17384,1.17327,1.17363,241
2025-09-11 18:40:00+00:00,1.17363,1.17378,1.17337,1.17339,226
2025-09-11 18:45:00+00:00,1.17339,1.17343,1.17302,1.17315,236
2025-09-11 18:50:00+00:00,1.17311,1.17336,1.17310,1.17317,229
2025-09-11 18:55:00+00:00,1.17319,1.17341,1.17302,1.17315,217


Rows: 6104


## 4) Estratégia simples: Crossover de Médias
- MA rápida: 20
- MA lenta: 50

Sinal:
- Compra quando `MA20` cruza acima de `MA50`
- Venda quando `MA20` cruza abaixo de `MA50`

Usaremos `AccountHandler` e `PositionHandler` para simular ordens/posições.


In [6]:
df = ohlc.copy()
df['ma_fast'] = df['close'].rolling(20).mean()
df['ma_slow'] = df['close'].rolling(50).mean()
df = df.dropna().copy()
display(df[['close','ma_fast','ma_slow']].head())


,close,ma_fast,ma_slow
time,,,
2025-09-11 22:40:00+00:00,1.17366,1.173511,1.173479
2025-09-11 22:45:00+00:00,1.17375,1.173529,1.173482
2025-09-11 22:50:00+00:00,1.17371,1.173547,1.173488
2025-09-11 22:55:00+00:00,1.17367,1.173571,1.173499
2025-09-11 23:00:00+00:00,1.17369,1.173590,1.173509


In [7]:
# Inicializa conta/posições para o backtest local
account_handler = AccountHandler(base_currency="USD", initial_balance=10_000, leverage=100)
position_handler = PositionHandler(account_handler=account_handler)

def close_all_active_positions(curr_price, curr_time):
    for pos in position_handler.positions[:]:
        if pos.status == PositionStatus.ACTIVE:
            position_handler.close_position(
                price=curr_price,
                quantity=pos.quantity,
                close_time=curr_time,
                reason="Signal flip",
                position=pos,
            )

prev_signal = 0  # -1 short, 0 neutro, 1 long
for ts, row in df.iterrows():
    price = float(row['close'])
    ts_iso = ts.isoformat()
    fast, slow = row['ma_fast'], row['ma_slow']
    signal = 1 if fast > slow else (-1 if fast < slow else prev_signal)

    # Flip de sinal -> fecha tudo, depois abre nova direção via ordem a mercado simulada
    if signal != prev_signal:
        # fecha posições abertas
        close_all_active_positions(curr_price=price, curr_time=pd.Timestamp(ts))
        # cria ordem da nova direção (usa preço=close para execução imediata nas regras do handler)
        if signal == 1:
            position_handler.create_order(OrderType.BUY,  price=price, quantity=1.0, symbol=symbol)
        elif signal == -1:
            position_handler.create_order(OrderType.SELL, price=price, quantity=1.0, symbol=symbol)

    # atualiza handler (processa ordens/SL/TP)
    position_handler.update({
        "Close": price,
        "Timestamp": ts_iso,
    })
    prev_signal = signal

# Ao final, fecha qualquer posição ainda aberta no último preço
if len(position_handler.positions):
    last_ts = pd.Timestamp(df.index[-1])
    last_px = float(df['close'].iloc[-1])
    close_all_active_positions(curr_price=last_px, curr_time=last_ts)

print("Backtest finalizado.")


2025-10-11 15:38:11,790 - INFO - Order created: {'order_id': 1, 'type': <OrderType.BUY: 1>, 'price': 1.17366, 'quantity': 1.0, 'symbol': 'EURUSD', 'sl': None, 'tp': None, 'status': <OrderStatus.PENDING: 1>, 'max_time_active': None, 'creation_time': Timestamp('2025-10-11 15:38:11.790783'), 'reason': None, 'position_reference': None}
2025-10-11 15:38:11,792 - INFO - Order 1 on EURUSD executed and converted to position #1.
2025-10-11 15:38:11,794 - INFO - Appended data to Pickle file: historical_orders.pkl
2025-10-11 15:38:11,794 - INFO - Appended Order {'order_id': 1, 'type': <OrderType.BUY: 1>, 'price': 1.17366, 'quantity': 1.0, 'symbol': 'EURUSD', 'sl': None, 'tp': None, 'status': <OrderStatus.EXECUTED: 2>, 'max_time_active': None, 'creation_time': Timestamp('2025-10-11 15:38:11.790783'), 'reason': None, 'position_reference': <algo_trading.position_handler.Position object at 0x0000024B85910290>} to historical_orders.pkl
2025-10-11 15:38:11,795 - INFO - Update completed. Pending orders 

Backtest finalizado.


## Resultados


In [8]:
closed = account_handler.get_closed_positions()
display(closed if isinstance(closed, pd.DataFrame) else pd.DataFrame(closed))
print("Realized PnL:", account_handler.realized_pnl)
print("Equity:", account_handler.equity)


,position_id,type,open_price,quantity,symbol,sl,tp,open_time,status,close_price,close_time,reason
0,1,PositionType.BUY,1.17366,1.0,EURUSD,None,None,2025-10-11 15:38:11.790783,PositionStatus.CLOSED,1.17347,2025-09-12 00:50:00+00:00,Signal flip
1,2,PositionType.SELL,1.17347,1.0,EURUSD,None,None,2025-10-11 15:38:11.812357,PositionStatus.CLOSED,1.17399,2025-09-12 02:00:00+00:00,Signal flip
2,3,PositionType.BUY,1.17399,1.0,EURUSD,None,None,2025-10-11 15:38:11.824645,PositionStatus.CLOSED,1.17295,2025-09-12 03:20:00+00:00,Signal flip
3,4,PositionType.SELL,1.17295,1.0,EURUSD,None,None,2025-10-11 15:38:11.839274,PositionStatus.CLOSED,1.17329,2025-09-12 08:40:00+00:00,Signal flip
4,5,PositionType.BUY,1.17329,1.0,EURUSD,None,None,2025-10-11 15:38:11.879378,PositionStatus.CLOSED,1.17148,2025-09-12 12:00:00+00:00,Signal flip
...,...,...,...,...,...,...,...,...,...,...,...,...
132,133,PositionType.BUY,1.15818,1.0,EURUSD,None,None,2025-10-11 15:38:16.021881,PositionStatus.CLOSED,1.15786,2025-10-10 13:30:00+00:00,Signal flip
133,134,PositionType.SELL,1.15786,1.0,EURUSD,None,None,2025-10-11 15:38:16.042518,PositionStatus.CLOSED,1.15950,2025-10-10 18:00:00+00:00,Signal flip
134,135,PositionType.BUY,1.15950,1.0,EURUSD,None,None,2025-10-11 15:38:16.084518,PositionStatus.CLOSED,1.16113,2025-10-10 21:30:00+00:00,Signal flip
135,136,PositionType.SELL,1.16113,1.0,EURUSD,None,None,2025-10-11 15:38:16.113933,PositionStatus.CLOSED,1.16144,2025-10-10 22:40:00+00:00,Signal flip


Realized PnL: 0.0
Equity: 10000
